In [ ]:
!pip install langid

In [ ]:
import pandas as pd
import re
import html
import nltk
import langid

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
pd.set_option('display.max_colwidth', None)

# 1. COLUMNS SELECTION + CLEANSE
**FOCUS**:
- Filter data to ENGLISH ONLY + add review_id to rollback if need
- Break reviews into sentences + Remove duplicate
- Probably separate labels data (in final.csv) and unlabel data (in ready_eda.csv) to use for our last model. -> reviews without sample data will be raw_unlabeled_pool_clean.csv

## 1. Prepare_lean_data will use for only eda_ready.csv

In [ ]:
def prepare_lean_data(file_path):
    df_raw = pd.read_csv(file_path)

    columns_to_keep = [
        'parent_asin', 'product_title', 'category', 'brand',
        'price_tier', 'price', 'average_rating', 'rating_number',
        'rating', 'review_text'
    ]

    df_lean = df_raw[columns_to_keep].copy()
    df_lean = df_lean.dropna(subset=['review_text'])
    df_lean = df_lean.reset_index(drop=True)
    df_lean.insert(0, 'review_id', range(1, len(df_lean) + 1))

    print(f" DROP NAN:   From: {len(df_raw)} rows -> Now: {len(df_lean)} rows.")
    return df_lean

In [ ]:
df_lean = prepare_lean_data('eda_ready.csv')
df_lean.head(2)

 DROP NAN:   From: 365944 rows -> Now: 365820 rows.


,review_id,parent_asin,product_title,category,brand,price_tier,price,average_rating,rating_number,rating,review_text
0,1,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,"There is nothing inside, just an empty shell. Which is not how it was described."
1,2,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now."


## 2. Use clean_text for df_clean (final.csv already went through clean text)

In [ ]:
def clean_text(text):
    if pd.isna(text): return ""
    text = html.unescape(str(text))

    # 1. placeholder + html tag
    text = re.sub(r"\[\[[a-z0-9_]+:[^\]]+\]\]", " ", text, flags=re.IGNORECASE)

    # 2. <br> or \n replace with .
    text = re.sub(r"<br\s*/?>", ". ", text, flags=re.IGNORECASE)
    text = re.sub(r"\n+", ". ", text)
    text = re.sub(r"<(?!3)[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    #  MARKDOWN & BULLET POINTS
    # From "## Features ##" thành ". Features ."  ~> can be clean later on
    text = re.sub(r'#+\s*(.*?)\s*#+', r'. \1. ', text)
    # Replace " - " to "." (only for "multi-core")
    text = re.sub(r'\s+-\s+', r'. ', text)
    text = re.sub(r'\s+\*\s+', r'. ', text)

    # Poor syntax ahhh (ex: "good.It is")
    text = re.sub(r'([a-z])([.!?])([A-Z])', r'\1\2 \3', text)

    # Space handle
    text = re.sub(r'\s+', ' ', text).strip()   #  Group
    text = re.sub(r'\s+\.', '.', text)         #  (" ." -> ".")
    text = re.sub(r'\.{2,}', '.', text)        #  ("..." -> ".")
    text = re.sub(r'\.\s*\.', '.', text)       #  (". ." -> ".")

    return text

In [ ]:
df_lean['review_text'] = df_lean['review_text'].apply(clean_text)

## 3. Only Take English reviews for training

In [ ]:
langid.set_languages(['en', 'es', 'fr', 'de', 'zh', 'ja'])

def safe_detect_langid(text):
    if pd.isna(text):
        return 'unknown'

    text_str = str(text).strip()

    # Too short (< 3 char) or full syntax = unknown
    if len(text_str) < 3 or not re.search('[a-zA-Z]', text_str):
        return 'unknown'

    try:
        lang, _ = langid.classify(text_str)
        return lang
    except Exception:
        return 'unknown'

In [ ]:
df_lean['language'] = df_lean['review_text'].apply(safe_detect_langid)
df_english = df_lean[df_lean['language'] == 'en'].copy()

In [ ]:
df_english = df_english.drop(columns=['language'])
print(f"FILTER ENGLISH:   From: {len(df_lean)} rows -> Now: {len(df_english)} rows.")

FILTER ENGLISH:   From: 365820 rows -> Now: 349714 rows.


## 4. Tach data trong final.csv ra khoi df_lean nham tranh data leakage

In [ ]:
labeled_data = pd.read_csv('final.csv')
sampled_contexts = labeled_data['review_context'].dropna().unique()
print(f"Number of reviews in final.csv: {len(sampled_contexts)}")

Number of reviews in final.csv: 479


In [ ]:
sampled_contexts_clean = set([clean_text(text) for text in sampled_contexts])

english_only = df_english.copy()

raw_text_column = 'review_text'
english_only['temp_fingerprint'] = english_only[raw_text_column].apply(clean_text)

df_no_leak = english_only[~english_only['temp_fingerprint'].isin(sampled_contexts_clean)].copy()
df_no_leak = df_no_leak.drop(columns=['temp_fingerprint'])

print(f"DROP SAMPLE DATA:   From: {len(english_only)} rows -> Now: {len(df_no_leak)} rows.")

DROP SAMPLE DATA:   From: 349714 rows -> Now: 347802 rows.


In [ ]:
test_contexts = labeled_data[labeled_data['review_context'].str.len() > 20]['review_context'].dropna().str.strip().unique()

leak_count = 0
training_texts = df_no_leak['review_text'].tolist()
training_texts_joined = " ||| ".join(training_texts)

for ctx in test_contexts:
    query = clean_text(ctx)
    if query in training_texts_joined:
        print(f"LEAKAGE: {query}")
        leak_count += 1

if leak_count == 0:
    print("PASS")
else:
    print(f"Leak {leak_count} reviews ")

LEAKAGE: Picture quality is great
LEAKAGE: I like everything about it.
LEAKAGE: This is a very good laptop for the price.
LEAKAGE: Was a Christmas present
LEAKAGE: Great value for price
LEAKAGE: Great little computer
Leak 6 reviews 


With the 6 very “generic” sentences above, we can confirm that the data is not leaked. It is simply that these 6 sentences are very generic, so they happened to match.

We want to keep them in the web app because they will be very useful for checking product aspects, but duplicates with the training data will be dropped.

## ***ALL IN ALL***:
- final.csv : sample data
- no_leak.csv: without sample data, havent split into sentences yet

# SPLITING REVIEW INTO SENTENCES

In [ ]:
def explode_reviews_to_sentences(df, text_column='review_text'):
    temp_df = df.copy()

    def split_to_sentences(text):
        if pd.isna(text) or text == "":
            return []
        return nltk.sent_tokenize(str(text))


    temp_df['sentence'] = temp_df[text_column].apply(split_to_sentences)
    df_exploded = temp_df.explode('sentence')
    df_exploded = df_exploded[df_exploded['sentence'].str.strip().str.len() > 5]
    df_exploded = df_exploded.reset_index(drop=True)

    # quality of life ahhh
    cols = ['review_id', text_column, 'sentence'] + \
           [c for c in df_exploded.columns if c not in ['review_id', text_column, 'sentence']]

    return df_exploded[cols]

In [ ]:
to_sentences = explode_reviews_to_sentences(df_no_leak, text_column='review_text')
print(f"From {len(df_no_leak)} in df_no_leak, to {len(to_sentences)} sentences.")

From 347802 in df_no_leak, to 1627364 sentences.


In [ ]:
to_sentences

,review_id,review_text,sentence,parent_asin,product_title,category,brand,price_tier,price,average_rating,rating_number,rating
0,1,"There is nothing inside, just an empty shell. Which is not how it was described.","There is nothing inside, just an empty shell.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
1,1,"There is nothing inside, just an empty shell. Which is not how it was described.",Which is not how it was described.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
2,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",The title implied I would receive a desktop computer with a said number of computer specs in the title.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
3,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",All I received was a case for a desktop computer.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
4,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.","There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1627359,365819,"The computer took a week to arrive and was well packed to prevent damage. Opening the box up revealed a larger all aluminum boxed computer than I expected but slimmer. Front and back were covered in sockets to provide just about any modern connection you could think of. I plugged it in to my 85"" tv as a monitor and added usb dongles for a mouse and keyboard. The Tv quickly recognized it and brought up windows, previously set up with administrator as user name and no password needed. All I needed this for was streaming TV from the web so I loaded Chrome which synchronized with my other computers and then I added a vpn to allow me to watch USTVGO giving me every channel for free, that my cable company wanted a fortune every month for. The main problem I had with a previous mini computer similarly priced, was it shutting down when left on for long periods, like overnight. This one has now been running continuously for several weeks. I can watch Amazon 4k videos with no waiting while it buffers, its just super smooth. Anything less than 4 k is just childs play for it. I was concerned that it felt quite warm to the touch but my infra red scanner told me it was only 95 degrees F. I stopped worrying about that. The only problem I did have was that the solid case tended to reduce the range of my mouse and keyboard. I wan

## WEBAPP DATA EXPORT

In [ ]:
web_app = explode_reviews_to_sentences(df_english, text_column='review_text')
print(f"From {len(df_english)} in raw_unlabeled_pool_clean, to {len(df_english)} sentences.")

From 349714 in raw_unlabeled_pool_clean, to 349714 sentences.


In [ ]:
web_app

,review_id,review_text,sentence,parent_asin,product_title,category,brand,price_tier,price,average_rating,rating_number,rating
0,1,"There is nothing inside, just an empty shell. Which is not how it was described.","There is nothing inside, just an empty shell.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
1,1,"There is nothing inside, just an empty shell. Which is not how it was described.",Which is not how it was described.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
2,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",The title implied I would receive a desktop computer with a said number of computer specs in the title.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
3,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",All I received was a case for a desktop computer.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
4,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.","There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1631351,365819,"The computer took a week to arrive and was well packed to prevent damage. Opening the box up revealed a larger all aluminum boxed computer than I expected but slimmer. Front and back were covered in sockets to provide just about any modern connection you could think of. I plugged it in to my 85"" tv as a monitor and added usb dongles for a mouse and keyboard. The Tv quickly recognized it and brought up windows, previously set up with administrator as user name and no password needed. All I needed this for was streaming TV from the web so I loaded Chrome which synchronized with my other computers and then I added a vpn to allow me to watch USTVGO giving me every channel for free, that my cable company wanted a fortune every month for. The main problem I had with a previous mini computer similarly priced, was it shutting down when left on for long periods, like overnight. This one has now been running continuously for several weeks. I can watch Amazon 4k videos with no waiting while it buffers, its just super smooth. Anything less than 4 k is just childs play for it. I was concerned that it felt quite warm to the touch but my infra red scanner told me it was only 95 degrees F. I stopped worrying about that. The only problem I did have was that the solid case tended to reduce the range of my mouse and keyboard. I wan

# DATA FOR MODEL

In [ ]:
def clean_text_for_model(text):
    if pd.isna(text): return ""
    text = clean_text(text).lower()
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
to_sentences['sentence_for_model'] = to_sentences['sentence'].apply(clean_text_for_model)
web_app['sentence_for_model'] = web_app['sentence'].apply(clean_text_for_model)

In [ ]:
to_sentences

,review_id,review_text,sentence,parent_asin,product_title,category,brand,price_tier,price,average_rating,rating_number,rating,sentence_for_model
0,1,"There is nothing inside, just an empty shell. Which is not how it was described.","There is nothing inside, just an empty shell.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,"there is nothing inside, just an empty shell."
1,1,"There is nothing inside, just an empty shell. Which is not how it was described.",Which is not how it was described.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,which is not how it was described.
2,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",The title implied I would receive a desktop computer with a said number of computer specs in the title.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,the title implied i would receive a desktop computer with a said number of computer specs in the title.
3,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",All I received was a case for a desktop computer.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,all i received was a case for a desktop computer.
4,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.","There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,"there were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1627359,365819,"The computer took a week to arrive and was well packed to prevent damage. Opening the box up revealed a larger all aluminum boxed computer than I expected but slimmer. Front and back were covered in sockets to provide just about any modern connection you could think of. I plugged it in to my 85"" tv as a monitor and added usb dongles for a mouse and keyboard. The Tv quickly recognized it and brought up windows, previously set up with administrator as user name and no password needed. All I needed this for was streaming TV from the web so I loaded Chrome which synchronized with my other computers and then I added a vpn to allow me to watch USTVGO giving me every channel for free, that my cable company wanted a fortune every month for. The main problem I had with a previous mini computer similarly priced, was it shutting down when left on for long periods, like overnight. This one has now been running continuously fo

In [ ]:
web_app

,review_id,review_text,sentence,parent_asin,product_title,category,brand,price_tier,price,average_rating,rating_number,rating,sentence_for_model
0,1,"There is nothing inside, just an empty shell. Which is not how it was described.","There is nothing inside, just an empty shell.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,"there is nothing inside, just an empty shell."
1,1,"There is nothing inside, just an empty shell. Which is not how it was described.",Which is not how it was described.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,which is not how it was described.
2,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",The title implied I would receive a desktop computer with a said number of computer specs in the title.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,the title implied i would receive a desktop computer with a said number of computer specs in the title.
3,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.",All I received was a case for a desktop computer.,B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,all i received was a case for a desktop computer.
4,2,"The title implied I would receive a desktop computer with a said number of computer specs in the title. All I received was a case for a desktop computer. There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities. Please refund me my money now.","There were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities.",B06XWPG52Q,*2017* Desktop Computer Intel Kaby Lake 3.6GHz 128GB SSD 3D NAND 8 GB RAM DDR4 HDMI USB 3.0 Windows 10 Pro Office 2016 (Black Orange),desktop,unknown,Unknown,NaN,1.0,6,1,"there were only electrical wires and built in fans that came with the case without the motherboard, ram, cpu, hard drive, and other necessities."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1631351,365819,"The computer took a week to arrive and was well packed to prevent damage. Opening the box up revealed a larger all aluminum boxed computer than I expected but slimmer. Front and back were covered in sockets to provide just about any modern connection you could think of. I plugged it in to my 85"" tv as a monitor and added usb dongles for a mouse and keyboard. The Tv quickly recognized it and brought up windows, previously set up with administrator as user name and no password needed. All I needed this for was streaming TV from the web so I loaded Chrome which synchronized with my other computers and then I added a vpn to allow me to watch USTVGO giving me every channel for free, that my cable company wanted a fortune every month for. The main problem I had with a previous mini computer similarly priced, was it shutting down when left on for long periods, like overnight. This one has now been running continuously fo

In [ ]:
final_web_app = web_app[web_app['sentence_for_model'].str.len() > 1]
final_web_app.to_csv('web_app.csv', index=False)

In [ ]:
inference_df = to_sentences[['sentence_for_model']].copy()
inference_df = inference_df[inference_df['sentence_for_model'].str.len() > 1]
inference_df = inference_df.drop_duplicates(subset=['sentence_for_model'])
inference_df.to_csv('final_nonlabel.csv', index=False)
print(f"final_nonlabel: {len(inference_df)} rows.")

final_nonlabel: 1485415 rows.
